In [ ]:
import solid2
from IPython.display import Image
import fuselage_variants as fv
import fuselage_splode as fs

# Everything in this notebook writes here, including the full-sweep cells below.
# Do not point this at variant_output: these cells are for poking at single parts,
# and a stray Run All would overwrite a sweep that takes hours to reproduce.
OUTPUT_DIR = 'test_fuse_output'

# Parameters arrive as dataclasses (Parameters, NoseParameters and the groups
# below them), not dicts -- so it is dp.panel.offset, never dp["panel"]["offset"].
# See doc/design/bulkhead.md and IP-GEO-16.

In [ ]:
fs.get_params()

In [ ]:
## Test a single boom bulkhead generator

csv_files = fv.axes('panel_variants.csv', 'bulkhead_size_variants.csv', 'boom_bulkhead_type_variants.csv')
param_axes = fv.read_all_param_axes(csv_files)
all_combinations = fv.flatten_param_space(param_axes)

printer_settings = fv.null_printer_settings()
printer_settings.extrusion_width = 0.6

params = all_combinations[7]
print("params = " + str(params))

FX = 1.0

params["panel_thickness_mm"] = 3/16*25.4
params["panel_name"] = "3_16in"
params["make_vert_web"] = True
params["make_lower_web"] = False
params["y_position"] = 0.2
params["z_position"] = 0.2
params["panel_is_metric"] = False
params["panel_thickness_32"] = 6


dp_bulk = fv.derived_parameters(params["U"], FX, params, printer_settings, True)

print("dp_bulk = " + str(dp_bulk))
print("panel_offset = " + str(dp_bulk.panel.offset))

is_valid_bulk = fv.bulkhead_validity_check(dp_bulk)
print("is_valid_bulk = " + str(is_valid_bulk))

if is_valid_bulk:

    filename = fv.generate_fuselage_boom_bulkhead_variant_filename_from_params(dp_bulk)
    fv.boom_bulkhead_render(dp_bulk, OUTPUT_DIR, filename)

In [ ]:
## Test a single nose generator
#
# The cowl axes are nose_size_variants.csv (per-U dimensions) and
# nose_type_variants.csv (which names a JSON shape file). The old
# is_nose_cowl/is_nose_nose/is_nose_plate columns are gone -- which of the three
# parts exist is now a property of the parameter file, read as dp.nose.active
# and dp.plate.active.

csv_files = fv.axes('nose_size_variants.csv', 'nose_type_variants.csv')
param_axes = fv.read_all_param_axes(csv_files)
all_combinations = fv.flatten_param_space(param_axes)

printer_settings = fv.null_printer_settings()
printer_settings.extrusion_width = 0.6

FX = 1.0
params = all_combinations[2]
print("params = " + str(params))

U = params["U"]
dp = fv.derived_cowl_parameters(U, FX, params, printer_settings)
print("dp = " + str(dp))

# (is_nose_cowl, is_nose_nose, is_nose_plate) -- the cowl always exists, the tip
# and the plate only if the parameter file says so.
wanted = [(True, False, False)]
if dp.nose.active:
    wanted.append((False, True, False))
if dp.plate.active:
    wanted.append((False, False, True))

for (is_cowl, is_nose, is_plate) in wanted:
    filename = fv.generate_fuselage_nose_variant_filename_from_params(
        U, dp, is_cowl, is_nose, is_plate)
    print("rendering " + filename)
    fv.nose_render(U, dp, OUTPUT_DIR, filename, is_cowl, is_nose, is_plate)

In [ ]:
## Test a single bulkhead generator

csv_files = fv.axes('panel_variants.csv', 'bulkhead_type_variants.csv', 'bulkhead_size_variants.csv')
param_axes = fv.read_all_param_axes(csv_files)
all_combinations = fv.flatten_param_space(param_axes)

printer_settings = fv.null_printer_settings()
printer_settings.extrusion_width = 0.6

params = all_combinations[7]
print("params = " + str(params))

FX = 1.0

dp_bulk = fv.derived_parameters(params["U"], FX, params, printer_settings, True)
dp_corn = fv.derived_parameters(params["U"], FX, params, printer_settings, False)

print("dp_bulk = " + str(dp_bulk))
print("panel_offset = " + str(dp_bulk.panel.offset))

is_valid_bulk = fv.bulkhead_validity_check(dp_bulk)
print("is_valid_bulk = " + str(is_valid_bulk))

if is_valid_bulk:

    filename = fv.generate_fuselage_bulkhead_variant_filename_from_params(dp_bulk)
    fv.bulkhead_render(dp_bulk, OUTPUT_DIR, filename)


print("dp_corn = " + str(dp_corn))
print("panel_offset = " + str(dp_corn.panel.offset))

is_valid_corn = fv.corner_validity_check(dp_corn)
print("is_valid_corn = " + str(is_valid_corn))

if is_valid_corn:

    filename = fv.generate_fuselage_corner_variant_filename_from_params(dp_corn)
    fv.corner_render(dp_corn, OUTPUT_DIR, filename)

In [ ]:
## Test a single corner generator at a chosen U / FX

csv_files = fv.axes('panel_variants.csv', 'bulkhead_size_variants.csv', 'corner_size_variants.csv')
param_axes = fv.read_all_param_axes(csv_files)
all_combinations = fv.flatten_param_space(param_axes)

printer_settings = fv.null_printer_settings()
printer_settings.extrusion_width = 0.6

params = all_combinations[0]

params["U"] = 4.0
params["FX"] = 1.0

print("params = ")
print(params)

dp = fv.derived_parameters(params["U"], params["FX"], params, printer_settings, False)

print("dp = ")
print(dp)

print("panel_offset = ")
print(dp.panel.offset)

is_valid = fv.corner_validity_check(dp)
print(is_valid)

if is_valid:

    filename = fv.generate_fuselage_corner_variant_filename_from_params(dp)
    fv.corner_render(dp, OUTPUT_DIR, filename)

In [ ]:
csv_files = fv.axes('panel_variants.csv', 'bulkhead_type_variants.csv', 'bulkhead_size_variants.csv')
param_axes = fv.read_all_param_axes(csv_files)

printer_settings = fv.null_printer_settings()
all_combinations = fv.flatten_param_space(param_axes)

print(all_combinations[40])

# is_bulkhead is required: it decides the greeble tolerance (the bulkhead's post
# is nominal, the corner's bore carries the clearance) and whether the bulkhead
# type fields are populated.
fv.derived_parameters(1, 1, all_combinations[40], printer_settings, True)

In [ ]:
# Full sweeps. These take hours and are better run from the CLI, which has
# parallel rendering, --resume and preview control:
#     uv run python src/Fuselage/tools/fuselage_variants.py --help
# They write to OUTPUT_DIR (test_fuse_output), never to variant_output.
fv.run_corner_parametric_sweep(fv.axes('panel_variants.csv', 'bulkhead_size_variants.csv', 'corner_size_variants.csv'), OUTPUT_DIR)

In [ ]:
fv.run_bulkhead_parametric_sweep(fv.axes('panel_variants.csv', 'bulkhead_type_variants.csv', 'bulkhead_size_variants.csv'), OUTPUT_DIR)

In [ ]:
fv.run_boom_bulkhead_parametric_sweep(fv.axes('panel_variants.csv', 'bulkhead_size_variants.csv', 'boom_bulkhead_type_variants.csv'), OUTPUT_DIR)

In [ ]:
# The cowl sweeps take the nose size axis, not the bulkhead one.
fv.run_nose_parametric_sweep(fv.axes('nose_size_variants.csv', 'nose_type_variants.csv'), OUTPUT_DIR)

In [ ]:
fv.run_tail_parametric_sweep(fv.axes('nose_size_variants.csv', 'tail_type_variants.csv'), OUTPUT_DIR)

In [ ]:
solid2.set_global_fn(0)
solid2.set_global_fa(5)
solid2.set_global_fs(0.5)

In [ ]:
# Import the geometry files themselves, not fuselage_geometry.scad. That file is
# a three-line aggregator of `include` lines; imported with use_not_include it
# exposes no modules at all, so fgeom.fuselage_corner did not exist.
#
# fv.scad_module() resolves against SCAD_DIR rather than the working directory,
# so these cells work wherever the kernel was started -- the same helper the
# sweep uses.
fcorner = fv.scad_module('fuselage_corner_geometry.scad')
fbulk   = fv.scad_module('fuselage_bulkhead_geometry.scad')
fboom   = fv.scad_module('fuselage_boom_bulkhead_geometry.scad')
fcowl   = fv.scad_module('cowl_geometry.scad')

In [ ]:
U = 1
FX = 1

DTF_thickness = 4.77

# These are based on the standard, don't change these
unit_width=100*U
unit_length=100*U*FX
corner_radius = 10*U
longeron_radius = 2*U

# printer settings
extrusion_width = 0.4
layer_height=0.2

bulkhead_thickness = 12
panel_thickness = DTF_thickness
panel_offset = 0
panel_overlap = 6
longeron_tolerance = 0.05
panel_tolerance = 0.1

# The corner's bore carries the whole greeble fit clearance; the bulkhead's post
# is nominal. Both walls are two extrusions at U=1.
greeble_tolerance = 0.05
greeble_thickness = 2*extrusion_width
greeble_nub_thickness = 2*extrusion_width

# Keyword arguments throughout: these signatures are long and same-typed, so a
# positional call silently shifts every argument if one is added. See IP-GEO-2.
corn = fcorner.fuselage_corner(
    U=U,
    unit_length=unit_length,
    bulkhead_thickness=bulkhead_thickness,
    corner_radius=corner_radius,
    panel_thickness=panel_thickness,
    panel_offset=panel_offset,
    panel_overlap=panel_overlap,
    panel_tolerance=panel_tolerance,
    longeron_radius=longeron_radius,
    longeron_tolerance=longeron_tolerance,
    greeble_thickness=greeble_thickness,
    greeble_nub_thickness=greeble_nub_thickness,
    greeble_tolerance=greeble_tolerance,
    extrusion_width=extrusion_width)

(scad_filename, stl_filename, png_filename) = fv.solid_render(corn, OUTPUT_DIR, 'tmp_corner.scad')
Image(filename=png_filename)

In [ ]:
FX = 1
U = 1.0

DTF_thickness = 4.77

# These are based on the standard, don't change these
unit_width=100*U
unit_length=100*U*FX
corner_radius = 10*U
longeron_radius = 2*U
bolt_offset=8*U

# printer settings
extrusion_width = 0.4
layer_height=0.2

# User parameters
is_interconnect = False
is_cowling = False
bulkhead_thickness = 6
panel_thickness = DTF_thickness
panel_offset = 0
panel_overlap = 4
panel_tolerance = 0.1
longeron_tolerance = 0.05
# bolt_hole_radius=4.3/2
bolt_hole_radius=5.33/2
bolt_thickness=3

# Half-angle: the mouth the longeron snaps in through is twice this.
greeble_opening_angle = 35
# Zero on the bulkhead -- its greeble post is nominal and the corner's bore
# carries the clearance.
greeble_tolerance = 0.0
greeble_thickness = 2*extrusion_width
greeble_nub_thickness = 2*extrusion_width

plate_thickness=4*layer_height
web_fillet_radius=2
web_width=3
flange_fillet_radius=2
flange_thickness=2*extrusion_width
flange_chamfer=1

cowl_flange_height=0
cowl_flange_tolerance=0

bulk = fbulk.bulkhead_section_full(
    is_interconnect=is_interconnect,
    is_cowling=is_cowling,
    unit_width=unit_width,
    # No unit_length: a bulkhead is independent of bay length (IP-GEO-23).
    bulkhead_thickness=bulkhead_thickness,
    corner_radius=corner_radius,
    panel_thickness=panel_thickness,
    panel_offset=panel_offset,
    panel_overlap=panel_overlap,
    panel_tolerance=panel_tolerance,
    longeron_radius=longeron_radius,
    longeron_tolerance=longeron_tolerance,
    bolt_hole_radius=bolt_hole_radius,
    bolt_thickness=bolt_thickness,
    bolt_offset=bolt_offset,
    greeble_opening_angle=greeble_opening_angle,
    greeble_thickness=greeble_thickness,
    greeble_nub_thickness=greeble_nub_thickness,
    # No greeble_tolerance: the bulkhead post is nominal by construction
    plate_thickness=plate_thickness,
    web_fillet_radius=web_fillet_radius,
    web_width=web_width,
    flange_fillet_radius=flange_fillet_radius,
    flange_thickness=flange_thickness,
    flange_chamfer=flange_chamfer,
    cowl_flange_height=cowl_flange_height,
    cowl_flange_tolerance=cowl_flange_tolerance,
    extrusion_width=extrusion_width)

(scad_filename, stl_filename, png_filename) = fv.solid_render(bulk, OUTPUT_DIR, 'tmp_bulk.scad')
Image(filename=png_filename)

In [ ]:
FX = 1
U = 1

DTF_thickness = 4.77

# TODO:
#   Boom key orientation
#   External booms

# These are based on the standard, don't change these
corner_radius = 10*U
longeron_radius = 2*U
unit_length=100*U*FX
unit_width=100*U
bolt_offset=8*U
boom_diameter=8*U

# printer settings
extrusion_width = 0.4
layer_height=0.2

# User parameters
panel_thickness = DTF_thickness
panel_overlap = 4
panel_offset = 0
panel_tolerance = 0.1

longeron_tolerance = 0.05

bolt_hole_radius=4.3/2

web_fillet_radius=2
web_width=6

boom_bulkhead_thickness = 2
boom_tolerance = 0.2
boom_collet_thickness = 3
boom_key_width = 2
boom_key_height = 2
boom_key_radius = 0.5
boom_key_angle = 0
boom_key_web_width = 6
boom_make_vert_web = True
boom_make_lower_web = False

# derived boom positions
boom_z_position = unit_width/2 - 0.25*unit_width
boom_y_position = 0*unit_width

bulk_boom = fboom.boom_bulkhead(
    unit_width=unit_width,
    corner_radius=corner_radius,
    panel_thickness=panel_thickness,
    panel_offset=panel_offset,
    panel_overlap=panel_overlap,
    panel_tolerance=panel_tolerance,
    longeron_radius=longeron_radius,
    longeron_tolerance=longeron_tolerance,
    bolt_hole_radius=bolt_hole_radius,
    bolt_offset=bolt_offset,
    web_fillet_radius=web_fillet_radius,
    web_width=web_width,
    boom_diameter=boom_diameter,
    boom_bulkhead_thickness=boom_bulkhead_thickness,
    boom_y_position=boom_y_position,
    boom_z_position=boom_z_position,
    boom_collet_thickness=boom_collet_thickness,
    boom_key_width=boom_key_width,
    boom_key_height=boom_key_height,
    boom_key_radius=boom_key_radius,
    boom_key_angle=boom_key_angle,
    boom_key_web_width=boom_key_web_width,
    boom_tolerance=boom_tolerance,
    boom_make_vert_web=boom_make_vert_web,
    boom_make_lower_web=boom_make_lower_web)

(scad_filename, stl_filename, png_filename) = fv.solid_render(bulk_boom, OUTPUT_DIR, 'tmp_bulk_boom.scad')
Image(filename=png_filename)

In [ ]:
FX = 1
U = 1

DTF_thickness = 4.77

# These are based on the standard, don't change these
corner_radius = 10*U
longeron_radius = 2*U+0.15
unit_length=100*U*FX
unit_width=100*U
bolt_offset=8*U

# printer settings
extrusion_width = 0.4
layer_height=0.2

# User parameters
is_interconnect = False
is_cowling = True

bulkhead_thickness = 6

# A cowling bulkhead carries no panel -- the cowl closes that end instead.
panel_thickness = 0
panel_overlap = 0
panel_offset = 0
panel_tolerance = 0.0

longeron_tolerance = 0.05

#bolt_hole_radius=4.3/2
bolt_hole_radius=5.33/2
bolt_thickness=2

greeble_opening_angle = 35
greeble_tolerance = 0.0
greeble_thickness = 2*extrusion_width
greeble_nub_thickness = 2*extrusion_width

web_fillet_radius=2
web_width=3

flange_fillet_radius=2
flange_chamfer=1

cowl_flange_height=2
cowl_flange_tolerance=0.2

# The cowling flange is one extrusion thicker than the standard one.
flange_thickness=3*extrusion_width
plate_thickness=4*layer_height

bulk_cowl = fbulk.bulkhead_section_full(
    is_interconnect=is_interconnect,
    is_cowling=is_cowling,
    unit_width=unit_width,
    # No unit_length: a bulkhead is independent of bay length (IP-GEO-23).
    bulkhead_thickness=bulkhead_thickness,
    corner_radius=corner_radius,
    panel_thickness=panel_thickness,
    panel_offset=panel_offset,
    panel_overlap=panel_overlap,
    panel_tolerance=panel_tolerance,
    longeron_radius=longeron_radius,
    longeron_tolerance=longeron_tolerance,
    bolt_hole_radius=bolt_hole_radius,
    bolt_thickness=bolt_thickness,
    bolt_offset=bolt_offset,
    greeble_opening_angle=greeble_opening_angle,
    greeble_thickness=greeble_thickness,
    greeble_nub_thickness=greeble_nub_thickness,
    # No greeble_tolerance: the bulkhead post is nominal by construction
    plate_thickness=plate_thickness,
    web_fillet_radius=web_fillet_radius,
    web_width=web_width,
    flange_fillet_radius=flange_fillet_radius,
    flange_thickness=flange_thickness,
    flange_chamfer=flange_chamfer,
    cowl_flange_height=cowl_flange_height,
    cowl_flange_tolerance=cowl_flange_tolerance,
    extrusion_width=extrusion_width)

(scad_filename, stl_filename, png_filename) = fv.solid_render(bulk_cowl, OUTPUT_DIR, 'tmp_bulk_cowl.scad')
Image(filename=png_filename)

In [ ]:
U = 1

plate_diam = U*60
plate_tol = 0.1
plate_thickness = 0.8
plate_flange_width = 2
cone_angle = 35

nose_len = U*50
cut_len = U*6
unit_width = U*100
nose_flange_inset = 0.5
nose_flange_height = 1.0
plate_flange_height = 1.0

buttress_z_offset = U*2
buttress_r_start = U*0
buttress_r_end = U*6.6
buttress_r_inset = U*3
buttress_cut_thickness = 0.1

# oml_ref() adds the ../oml/ prefix. The import() lives in cowl_geometry.scad and
# OpenSCAD resolves it against the file containing the call, so a bare
# "vsp_nose.stl" does not resolve -- the same bug IP-GEO-18 fixed in the drivers.
oml_filename = fv.oml_ref("vsp_nose.stl")
oml_scale=1e-3
oml_length = 0.050
oml_offset_x=0
oml_reversed = False

nose_cowl = fcowl.nose_cowl(
    U=U,
    unit_width=unit_width,
    oml_filename=oml_filename,
    oml_scale=oml_scale,
    oml_length=oml_length,
    oml_offset_x=oml_offset_x,
    oml_reversed=oml_reversed,
    cut_len=cut_len,
    buttress_cut_thickness=buttress_cut_thickness,
    buttress_z_offset=buttress_z_offset,
    buttress_r_start=buttress_r_start,
    buttress_r_end=buttress_r_end,
    buttress_r_inset=buttress_r_inset,
    cone_angle=cone_angle)

(scad_filename, stl_filename, png_filename) = fv.solid_render(nose_cowl, OUTPUT_DIR, 'tmp_nose_cowl.scad')
Image(filename=png_filename)

In [ ]:
U = 1

plate_diam = U*60
plate_tol = 0.1
plate_thickness = 0.8
plate_flange_width = 2
cone_angle = 35

cut_len = U*6
unit_width = U*100
nose_flange_inset = 0.5
nose_flange_height = 1.0
plate_flange_height = 1.0

oml_filename = fv.oml_ref("vsp_nose.stl")
oml_scale=1e-3
oml_offset_x=0
oml_reversed = False

nose = fcowl.nose(
    U=U,
    unit_width=unit_width,
    oml_filename=oml_filename,
    oml_scale=oml_scale,
    oml_offset_x=oml_offset_x,
    oml_reversed=oml_reversed,
    cut_len=cut_len,
    nose_flange_height=nose_flange_height,
    nose_flange_inset=nose_flange_inset,
    plate_diam=plate_diam,
    plate_thickness=plate_thickness,
    plate_tol=plate_tol,
    cone_angle=cone_angle)

(scad_filename, stl_filename, png_filename) = fv.solid_render(nose, OUTPUT_DIR, 'tmp_nose.scad')
Image(filename=png_filename)

In [ ]:
U = 1

plate_diam = U*60
plate_thickness = 0.8
plate_flange_width = 2
plate_flange_height = 1.0
cone_angle = 35

nose_plate = fcowl.nose_plate(
    plate_diam=plate_diam,
    plate_thickness=plate_thickness,
    plate_flange_width=plate_flange_width,
    plate_flange_height=plate_flange_height,
    cone_angle=cone_angle)

(scad_filename, stl_filename, png_filename) = fv.solid_render(nose_plate, OUTPUT_DIR, 'tmp_nose_plate.scad')
Image(filename=png_filename)

In [ ]:
full_nose = solid2.rotate([0,-90,0]) (
    nose_plate
    + solid2.translate([0,0,-24])(nose)
    + solid2.translate([0,0,-30])(nose_cowl) 
    + solid2.translate([0,0,-100])(bulk_cowl)
)
(scad_filename, stl_filename, png_filename) = fv.solid_render(full_nose, 'test_fuse_output', 'tmp_full_nose.scad')
Image(filename=png_filename)

In [ ]:
U = 1

unit_width = U*100
cut_len = 0

cone_angle = 35

buttress_z_offset = U*2
buttress_r_inset = U*5
buttress_cut_thickness = 0.1

side_buttress_z_end = U*25
side_buttress_r_start = U*0
side_buttress_r_end = U*23.3   # U*32.1

top_buttress_z_end = U*3
top_buttress_r_start = U*0
top_buttress_r_end = U*0

top_diag_buttress_z_start = U*20
top_diag_buttress_depth = U*2

bottom_buttress_z_end = U*20
bottom_buttress_r_start = U*0
bottom_buttress_r_end = U*28.8  # U*38.3

oml_filename = fv.oml_ref("vsp_tail.stl")
oml_scale = 1e-3
oml_length = 0.1
oml_offset_x = -0.25
# The tail OML is built nose-first, so it is imported reversed. Omitting this
# left the argument undefined and every buttress dimension shifted by one.
oml_reversed = True

tail_cowl = fcowl.tail_cowl(
    U=U,
    unit_width=unit_width,
    oml_filename=oml_filename,
    oml_scale=oml_scale,
    oml_length=oml_length,
    oml_offset_x=oml_offset_x,
    oml_reversed=oml_reversed,
    cut_len=cut_len,
    buttress_cut_thickness=buttress_cut_thickness,
    buttress_z_offset=buttress_z_offset,
    buttress_r_inset=buttress_r_inset,
    side_buttress_z_end=side_buttress_z_end,
    side_buttress_r_start=side_buttress_r_start,
    side_buttress_r_end=side_buttress_r_end,
    top_buttress_z_end=top_buttress_z_end,
    top_buttress_r_start=top_buttress_r_start,
    top_buttress_r_end=top_buttress_r_end,
    bottom_buttress_z_end=bottom_buttress_z_end,
    bottom_buttress_r_start=bottom_buttress_r_start,
    bottom_buttress_r_end=bottom_buttress_r_end,
    top_diag_buttress_depth=top_diag_buttress_depth,
    top_diag_buttress_z_start=top_diag_buttress_z_start,
    cone_angle=cone_angle)

(scad_filename, stl_filename, png_filename) = fv.solid_render(tail_cowl, OUTPUT_DIR, 'tmp_tail_cowl.scad')
Image(filename=png_filename)

In [ ]:
full_tail = solid2.rotate([0,-90,0]) (solid2.rotate([0,180,0])(bulk_cowl + solid2.translate([0,0,120])(tail_cowl) ) )
             
(scad_filename, stl_filename, png_filename) = fv.solid_render(full_tail, 'test_fuse_output', 'tmp_full_tail.scad')
Image(filename=png_filename)

In [ ]:
def unit_frame():
    """One bay: two bulkheads and four corners.

    Self-contained. The previous version read cowl_flange_height and
    cowl_flange_tolerance from module scope, so it silently inherited whatever
    the cowling cell above happened to leave behind -- and produced a cowling
    lip on a plain bay if that cell had been run.
    """
    U = 1
    FX = 1

    DTF_thickness = 4.77

    # These are based on the standard, don't change these
    unit_width=100*U
    unit_length=100*U*FX
    corner_radius = 10*U
    longeron_radius = 2*U
    bolt_offset=8*U

    # printer settings
    extrusion_width = 0.4
    layer_height=0.2

    # User parameters
    bulkhead_thickness = 6
    panel_thickness = DTF_thickness
    panel_offset = 0
    panel_overlap = 4
    panel_tolerance = 0.1

    longeron_tolerance = 0.05

    # bolt_hole_radius=4.3/2
    bolt_hole_radius=5.33/2
    bolt_thickness=3

    greeble_opening_angle = 35
    greeble_thickness = 2*extrusion_width
    greeble_nub_thickness = 2*extrusion_width

    plate_thickness=4*layer_height

    web_fillet_radius=2
    web_width=3

    flange_fillet_radius=2
    flange_thickness=2*extrusion_width
    flange_chamfer=1

    # Not a cowling bay.
    cowl_flange_height=0
    cowl_flange_tolerance=0

    # The corner's bore takes the clearance; the bulkhead's greeble post is nominal.
    corn = fcorner.fuselage_corner(
        U=U,
        unit_length=unit_length,
        bulkhead_thickness=bulkhead_thickness,
        corner_radius=corner_radius,
        panel_thickness=panel_thickness,
        panel_offset=panel_offset,
        panel_overlap=panel_overlap,
        panel_tolerance=panel_tolerance,
        longeron_radius=longeron_radius,
        longeron_tolerance=longeron_tolerance,
        greeble_thickness=greeble_thickness,
        greeble_nub_thickness=greeble_nub_thickness,
        greeble_tolerance=0.05,
        extrusion_width=extrusion_width)

    bulk = fbulk.bulkhead_section_full(
        is_interconnect=False,
        is_cowling=False,
        unit_width=unit_width,
        # No unit_length: a bulkhead is independent of bay length (IP-GEO-23).
        bulkhead_thickness=bulkhead_thickness,
        corner_radius=corner_radius,
        panel_thickness=panel_thickness,
        panel_offset=panel_offset,
        panel_overlap=panel_overlap,
        panel_tolerance=panel_tolerance,
        longeron_radius=longeron_radius,
        longeron_tolerance=longeron_tolerance,
        bolt_hole_radius=bolt_hole_radius,
        bolt_thickness=bolt_thickness,
        bolt_offset=bolt_offset,
        greeble_opening_angle=greeble_opening_angle,
        greeble_thickness=greeble_thickness,
        greeble_nub_thickness=greeble_nub_thickness,
        # No greeble_tolerance: the bulkhead post is nominal by construction
        plate_thickness=plate_thickness,
        web_fillet_radius=web_fillet_radius,
        web_width=web_width,
        flange_fillet_radius=flange_fillet_radius,
        flange_thickness=flange_thickness,
        flange_chamfer=flange_chamfer,
        cowl_flange_height=cowl_flange_height,
        cowl_flange_tolerance=cowl_flange_tolerance,
        extrusion_width=extrusion_width)

    frame = bulk
    frame += solid2.translate([0,0,unit_length]) ( solid2.mirror([0,0,-1])(bulk) )
    frame += solid2.translate([unit_width/2-corner_radius, unit_width/2-corner_radius, 0]) (corn)
    frame += solid2.mirror([-1,0,0])(solid2.translate([unit_width/2-corner_radius, unit_width/2-corner_radius, 0]) (corn))
    frame += solid2.mirror([0,-1,0])(solid2.translate([unit_width/2-corner_radius, unit_width/2-corner_radius, 0]) (corn))
    frame += solid2.mirror([-1,-1,0])(solid2.translate([unit_width/2-corner_radius, unit_width/2-corner_radius, 0]) (corn))
    frame = solid2.rotate(0,-90,0)(frame)

    return frame

In [ ]:
# Needs the nose, tail, boom and cowl cells above to have been run -- they define
# full_nose, full_tail and bulk_boom.
frame = unit_frame()

(scad_filename, stl_filename, png_filename) = fv.solid_render(full_nose
                                                           + solid2.translate([100+100+10,0,0])(frame)
                                                           + solid2.translate([100+200+20,0,0])(frame)
                                                           + solid2.rotate([90,0,0])(solid2.translate([+100+200+30,0,0])(solid2.rotate([0,-90,0])(bulk_boom)))
                                                           + solid2.translate([+100+300+40,0,0])(frame)
                                                           + solid2.rotate([90,0,0])(solid2.translate([+100+300+50,0,0])(solid2.rotate([0,-90,0])(bulk_boom)))
                                                           + solid2.translate([+100+300+60,0,0])(full_tail)
                                                           , OUTPUT_DIR, 'tmp_splode.scad')
Image(filename=png_filename)